<a href="https://colab.research.google.com/github/hristinagjorgjievska/Natural-Language-Processing/blob/master/lab2/lab2_homework.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install evaluate
!pip install bert_score
!pip install "transformers<5"

In [ ]:
import pandas as pd

In [ ]:
data = pd.read_csv('sarcasm.txt', sep='\t')
data = data.sample(frac=1, random_state=42, ignore_index=True)
data.sample(5)

,Style 1,Style 2
186,Update: they’ve finally closed the campus whic...,"Although the uni has now closed the campus, I ..."
182,yeah your girl is fine but does she pass out w...,Passing out while giving blood is not attractive
280,Love being called the devil for taking away so...,“It was not enjoyable to be called the devil f...
406,@abhiishekkkkkk @FCBGox Did you even watch a s...,"You didn't watch Varane last season, if you di..."
251,never drinking again,I had a bad experience drinking today


In [ ]:
train_size = int(0.7 * len(data))
val_size = int(0.15 * len(data))

train_data = data[:train_size]
val_data = data[train_size: train_size + val_size]
test_data = data[train_size + val_size:]

In [ ]:
train_sarcasm = train_data['Style 1'].tolist()
train_sincere = train_data['Style 2'].tolist()

val_sarcasm = val_data['Style 1'].tolist()
val_sincere = val_data['Style 2'].tolist()

test_sarcasm = test_data['Style 1'].tolist()
test_sincere = test_data['Style 2'].tolist()

In [ ]:
device = 'cuda'

def train(model, train_loader, optimizer, epochs=5, device=device):
  model = model.to(device)
  model.train()

  for epoch in range(epochs):
    total_loss = 0.0
    for batch in train_loader:
      batch = {k: v.to(device) for k, v in batch.items()}
      loss = model(**batch).loss
      optimizer.zero_grad()
      loss.backward()
      optimizer.step()
      total_loss += loss.item()

    print(f'Epoch {epoch + 1}/{epochs}, Loss: {total_loss / len(train_loader):.4f}')

In [ ]:
from datasets import Dataset
from torch.utils.data import DataLoader
from transformers import DataCollatorForSeq2Seq

In [ ]:
def make_loader(tokenizer, model, batch_size):
  inputs = tokenizer(train_sarcasm, max_length=64, truncation=True)
  outputs = tokenizer(text_target=train_sincere, max_length=64, truncation=True)

  train_set = Dataset.from_dict({'input_ids': inputs['input_ids'],
                                 'attention_mask': inputs['attention_mask'],
                                 'labels': outputs['input_ids']})

  data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

  return DataLoader(train_set, batch_size=batch_size, shuffle=True, collate_fn=data_collator)

In [ ]:
from evaluate import load

In [ ]:
bleu = load('bleu')
bertscore = load('bertscore')

def translate_all(model, tokenizer, prompts):
  model = model.to(device)
  model.eval()

  predictions = []
  for prompt in prompts:
    tokens = tokenizer([prompt], return_tensors='pt', truncation=True).to(device)
    output = model.generate(**tokens, max_length=80)
    predictions.append(tokenizer.decode(output[0], skip_special_tokens=True))

  return predictions

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
def score(predictions, references):
  bleu_score = bleu.compute(predictions=predictions, references=references)['bleu']

  f1_score = bertscore.compute(predictions=predictions, references=references, model_type='distilroberta-base')['f1']

  return bleu_score, sum(f1_score) / len(f1_score)

In [ ]:
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

In [ ]:
results = []
max_score = -1

for lr in [0.0001, 0.00001]:
  for epochs in [2,5]:
    for batch_size in [8, 16]:
      print(f'- lr = {lr}, epochs = {epochs}, batch_size = {batch_size} -')

      tokenizer = AutoTokenizer.from_pretrained('t5-small')
      model = AutoModelForSeq2SeqLM.from_pretrained('t5-small')
      optimizer = AdamW(model.parameters(), lr=lr)
      train(model, make_loader(tokenizer, model, batch_size), optimizer, epochs, device=device)

      bleu_score, bert_f1_score = score(translate_all(model, tokenizer, val_sarcasm), val_sincere)

      results.append({'lr': lr,
                     'epochs': epochs,
                     'batch_size': batch_size,
                     'bleu': bleu_score,
                     'BERTScore': bert_f1_score})

      if bert_f1_score > max_score:
        max_score = bert_f1_score

        model.save_pretrained('models/t5_small_best')
        tokenizer.save_pretrained('models/t5_small_best')

- lr = 0.0001, epochs = 2, batch_size = 8 -
Epoch 1/2, Loss: 3.3544
Epoch 2/2, Loss: 3.0338
- lr = 0.0001, epochs = 2, batch_size = 16 -
Epoch 1/2, Loss: 3.4392
Epoch 2/2, Loss: 3.1166


- lr = 0.0001, epochs = 5, batch_size = 8 -
Epoch 1/5, Loss: 3.3587
Epoch 2/5, Loss: 3.0703
Epoch 3/5, Loss: 2.9146
Epoch 4/5, Loss: 2.7685
Epoch 5/5, Loss: 2.7068
- lr = 0.0001, epochs = 5, batch_size = 16 -
Epoch 1/5, Loss: 3.4381
Epoch 2/5, Loss: 3.1397
Epoch 3/5, Loss: 2.9837
Epoch 4/5, Loss: 2.9230
Epoch 5/5, Loss: 2.8342
- lr = 1e-05, epochs = 2, batch_size = 8 -
Epoch 1/2, Loss: 3.7190
Epoch 2/2, Loss: 3.5364


- lr = 1e-05, epochs = 2, batch_size = 16 -
Epoch 1/2, Loss: 3.6991
Epoch 2/2, Loss: 3.5975


- lr = 1e-05, epochs = 5, batch_size = 8 -
Epoch 1/5, Loss: 3.6801
Epoch 2/5, Loss: 3.5761
Epoch 3/5, Loss: 3.4393
Epoch 4/5, Loss: 3.3585
Epoch 5/5, Loss: 3.3322


- lr = 1e-05, epochs = 5, batch_size = 16 -
Epoch 1/5, Loss: 3.7384
Epoch 2/5, Loss: 3.6439
Epoch 3/5, Loss: 3.5503
Epoch 4/5, Loss: 3.4604
Epoch 5/5, Loss: 3.4437


In [ ]:
pd.DataFrame(results).sort_values('BERTScore', ascending=False)

,lr,epochs,batch_size,bleu,BERTScore
2,0.00010,5,8,0.079259,0.894099
3,0.00010,5,16,0.079112,0.893790
0,0.00010,2,8,0.088121,0.893330
1,0.00010,2,16,0.077515,0.862779
4,0.00001,2,8,0.080011,0.826292
6,0.00001,5,8,0.083764,0.816549
5,0.00001,2,16,0.075421,0.807674
7,0.00001,5,16,0.064666,0.801023


In [ ]:
tokenizer = AutoTokenizer.from_pretrained('models/t5_small_best')
model = AutoModelForSeq2SeqLM.from_pretrained('models/t5_small_best')

fine_tuned = score(translate_all(model, tokenizer, test_sarcasm), test_sincere)

t5baseTokenizer = AutoTokenizer.from_pretrained('t5-base')
t5baseModel = AutoModelForSeq2SeqLM.from_pretrained('t5-base')

prompt = 'Rewrite the following sarcastic text in a sincere way:'

zero_prompts = [f'{prompt} {sentence}' for sentence in test_sarcasm]

zero_shot = score(translate_all(t5baseModel, t5baseTokenizer, zero_prompts), test_sincere)

config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


In [ ]:
from sentence_transformers import SentenceTransformer, util

In [32]:
embedding_model = SentenceTransformer('all-distilroberta-v1')
train_emb = embedding_model.encode(train_sarcasm, batch_size=64)
test_emb = embedding_model.encode(test_sarcasm, batch_size=64)
test_hits = util.semantic_search(test_emb, train_emb, top_k=3)

few_prompts = []
for i in range(len(test_sarcasm)):
    examples = ''
    for hit in test_hits[i]:
        j = hit['corpus_id']
        examples += f'Input: {train_sarcasm[j]} Output: {train_sincere[j]}\n'

    few_prompts.append(f'Examples:\n{examples}{prompt} {test_sarcasm[i]}\nAnswer: ')

few_shot = score(translate_all(t5baseModel, t5baseTokenizer, few_prompts), test_sincere)

In [33]:
pd.DataFrame([fine_tuned, zero_shot, few_shot], columns=['bleu', 'BERTScore'],
             index=['fine-tuned T5-small', 'zero-shot T5-base', 'few-shot T5-base'])

,bleu,BERTScore
fine-tuned T5-small,0.098985,0.896232
zero-shot T5-base,0.044632,0.815359
few-shot T5-base,0.017577,0.843642


## Одговор за ЗАДАЧА 1
1. Го тренирав t5-small со сите 8 комбинации и секој модел го оценив на валидациското множество со bleu и BERTScore. Ја одбрав комбинацијата lr=0.0001, epochs=5, batch_size=8, бидејќи има најдобар BERTScore=0.894(покажува колку значењето е слично, што е поважно од тоа дали зборовите се исти).
2. Најдобар е t5-small моделот што е трениран на нашето податочно множество (BERTScore = 0.896), бидејќи научил од примерите како од sarcastic да претвори во sincere реченица. <br>
3. t5-base без примери (zero-shot, BERTScore = 0.815) е најлош бидејќи не го разбира prompt-от.  <br>
4. Со дадени примери(few-shot, BERTScore = 0.844) е подобар, но често копира делови од примерите, затоа bleu е најнизок.

In [34]:
import torch
from torch.utils.data import Dataset as Dataset2

In [35]:
class SarcasmDataset(Dataset2):
  def __init__(self, encodings, labels):
    self.encodings = encodings
    self.labels = labels

  def __getitem__(self, idx):
    item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
    item['labels'] = torch.tensor(self.labels[idx])
    return item

  def __len__(self):
    return len(self.labels)

In [36]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [53]:

def calculate_metrics(true_labels, predicted_labels):
  return {'accuracy': accuracy_score(true_labels, predicted_labels),
          'precision': precision_score(true_labels, predicted_labels),
          'recall': recall_score(true_labels, predicted_labels),
          'f1_score': f1_score(true_labels, predicted_labels)}

def train_classifier(model_name, lr, epochs, batch_size, test_texts_list, test_labels_list):
  tokenizer = AutoTokenizer.from_pretrained(model_name)
  model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

  training_args = TrainingArguments(output_dir='output/', report_to='none', save_strategy='no', learning_rate=lr, num_train_epochs=epochs, per_device_train_batch_size=batch_size, logging_strategy='epoch', eval_strategy='epoch')
  eval_set = SarcasmDataset(tokenizer(test_texts_list, truncation=True), test_labels_list)
  trainer = Trainer(model=model,
                    args=training_args,
                    train_dataset=SarcasmDataset(tokenizer(train_texts, truncation=True), train_labels),
                    eval_dataset=eval_set,
                    processing_class=tokenizer)

  training_info = trainer.train()
  output = trainer.predict(eval_set)

  summary = {'model': model_name, 'lr': lr, 'epochs': epochs, 'batch_size': batch_size,
         'train_loss': training_info.training_loss, 'eval_loss': output.metrics['test_loss'],
         'time_seconds': training_info.metrics['train_runtime']}

  summary.update(calculate_metrics(test_labels_list, output.predictions.argmax(axis=1)))

  return summary

In [54]:
train_texts = train_data['Style 1'].tolist() + train_data['Style 2'].tolist()
val_texts = val_data['Style 1'].tolist() + val_data['Style 2'].tolist()
test_texts = test_data['Style 1'].tolist() + test_data['Style 2'].tolist()

In [55]:
train_labels = [1] * len(train_data) + [0] * len(train_data)
val_labels = [1] * len(val_data) + [0] * len(val_data)
test_labels = [1] * len(test_data) + [0] * len(test_data)

In [56]:
results2 = []
for model_name in ['roberta-base', 'distilbert-base-uncased']:
  results2.append(train_classifier(model_name, 0.00001, 2, 16, val_texts, val_labels))

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss
1,0.652400,0.548797
2,0.447000,0.414026


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss
1,0.606900,0.494630
2,0.427200,0.416766


In [57]:
pd.DataFrame(results2)

,model,lr,epochs,batch_size,train_loss,eval_loss,time_seconds,accuracy,precision,recall,f1_score
0,roberta-base,0.00001,2,16,0.549701,0.414026,24.8001,0.796117,0.827957,0.747573,0.785714
1,distilbert-base-uncased,0.00001,2,16,0.517032,0.416766,10.8863,0.825243,0.868132,0.766990,0.814433


In [68]:
best_model = 'distilbert-base-uncased'

results3 = []
for lr in [0.0001, 0.00001]:
  for epochs in [2,5]:
    for batch_size in [16, 32]:
      results3.append(train_classifier(best_model, lr, epochs, batch_size, val_texts, val_labels))

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss
1,0.423200,0.330258
2,0.181700,0.369158


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss
1,0.424200,0.304176
2,0.187600,0.352267


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss
1,0.462500,0.327208
2,0.208800,0.315536
3,0.084700,0.490277
4,0.039900,0.541147
5,0.008400,0.556713


... (6 outputs left out)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss
1,0.592000,0.449287
2,0.361200,0.342117
3,0.278500,0.330663
4,0.233300,0.326331
5,0.220400,0.328219


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss
1,0.641600,0.543130
2,0.458300,0.407120
3,0.350600,0.376990
4,0.302700,0.358421
5,0.284800,0.353769


In [69]:
results3 = pd.DataFrame(results3).sort_values('f1_score', ascending=False)
results3

,model,lr,epochs,batch_size,train_loss,eval_loss,time_seconds,accuracy,precision,recall,f1_score
2,distilbert-base-uncased,0.00010,5,16,0.160855,0.556713,26.6647,0.883495,0.907216,0.854369,0.880000
3,distilbert-base-uncased,0.00010,5,32,0.152870,0.566376,27.8114,0.873786,0.881188,0.864078,0.872549
0,distilbert-base-uncased,0.00010,2,16,0.302406,0.369158,12.8384,0.868932,0.865385,0.873786,0.869565
1,distilbert-base-uncased,0.00010,2,32,0.305870,0.352267,12.2045,0.868932,0.931818,0.796117,0.858639
6,distilbert-base-uncased,0.00001,5,16,0.337082,0.328219,27.1843,0.849515,0.891304,0.796117,0.841026
7,distilbert-base-uncased,0.00001,5,32,0.407584,0.353769,28.0445,0.839806,0.897727,0.766990,0.827225
5,distilbert-base-uncased,0.00001,2,32,0.602966,0.536431,11.4413,0.791262,0.767857,0.834951,0.800000
4,distilbert-base-uncased,0.00001,2,16,0.504925,0.406121,11.1914,0.815534,0.882353,0.728155,0.797872


In [70]:
best_combination = results3.iloc[0]
train_classifier(best_model,
                 best_combination['lr'],
                 int(best_combination['epochs']),
                 int(best_combination['batch_size']),
                 test_texts,
                 test_labels)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss
1,0.433200,0.341040
2,0.202300,0.366403
3,0.074800,0.682333
4,0.030200,0.676913
5,0.007100,0.675578


{'model': 'distilbert-base-uncased',
 'lr': np.float64(0.0001),
 'epochs': 5,
 'batch_size': 16,
 'train_loss': 0.14951591540555484,
 'eval_loss': 0.6755779385566711,
 'time_seconds': 27.0896,
 'accuracy': 0.8809523809523809,
 'precision': 0.9,
 'recall': 0.8571428571428571,
 'f1_score': 0.8780487804878049}

In [71]:
t5_tokenizer = AutoTokenizer.from_pretrained('t5-small')
t5_model = AutoModelForSeq2SeqLM.from_pretrained('t5-small')

question = 'Classify the following sentence into \'SARCASTIC\' or \'SINCERE\':'
examples = f'Input: {train_sarcasm[0]} Output: SARCASTIC\nInput: {train_sincere[0]} Output: SINCERE\n'

def promptingt5model(text):
    tokens = t5_tokenizer(text, return_tensors='pt', truncation=True, max_length=512)
    output_ids = t5_model.generate(tokens.input_ids, max_new_tokens=5)
    answer = t5_tokenizer.decode(output_ids[0], skip_special_tokens=True).lower()

    if 'sarcastic' in answer:
      return 1
    else:
       return 0

zero_shot = [promptingt5model(f'{question} {sentence}') for sentence in test_texts]
few_shot = [promptingt5model(f'Examples:\n{examples}{question} {sentence} Output: ') for sentence in test_texts]

In [72]:
pd.DataFrame([calculate_metrics(test_labels, zero_shot), calculate_metrics(test_labels, few_shot)],
             index=['zero-shot', 'few-shot'])

/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


,accuracy,precision,recall,f1_score
zero-shot,0.500000,0.000,0.000000,0.000000
few-shot,0.509524,0.625,0.047619,0.088496


## Задача 2

1. Ги тренирав RoBERTa и DistilBERT со исти хиперпараметри и ги споредив на валидациско. DistilBERT е подобар во сите метрики и тренира дупло побрзо, затоа го избрав него.
2. Функции на загуба: со поголема lr (0.0001) и поголем број на епохи, training loss паѓа најмногу (0.16), а метриките растат (f1=0.880). Со мала lr (0.00001) и 2 епохи, загубата останува висока (0.60) и f1 најнизок (околу 0.80).
3. Време и перформанси: со 5 епоги тренирањето трае дупло подолго, но затоа и моделот е подобар. Batch size скоро и не влијае на времето и резултатите.
4. Најдобра конфигурација: lr=0.0001, epochs=5, batch_size=16 бидејќи добивме најдобар f1=0.880 и accuracy=0.883.
5. Промптинг со t5-small: резултатите се значително полоши. Со zero-shot моделот ниеднаш не кажува 'sarcastic', (f1=0) а со few-shot многу ретко (f1=0.088). Ова се случува бидејќи t5-small е мал модел и не е учен да следи инструкции ниту ги видел нашите податоци. DistilBERT е трениран на нив и затоа е многу подобар.